In [1]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git'
REPO_DIR = Path(os.environ.get(
    'AGRIPLAN_REPO_DIR',
    '/content/IT3091---Machine-Learning-Assignment'
))
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository ready.')


Repository ready.


# Wijesinghe D.T.D. crop-yield preprocessing



In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAW_FILE = REPO_DIR / 'Raw Datasets' / 'researchData.xlsx'
PREPROCESSED_DIR = REPO_DIR / 'Wijesinghe.D.T.D-Crop Yield Preprocessing'
OUTPUT_DIR = REPO_DIR / 'Wijesinghe.D.T.D-Crop Yield Outputs'
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw = pd.read_excel(RAW_FILE)
required = ['District', 'Season', 'CropCategory', 'Crop', 'Year', 'Extent', 'Production']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

yield_data = raw[required].copy()
for column in ['Year', 'Extent', 'Production']:
    yield_data[column] = (
        yield_data[column].astype(str).str.replace(',', '', regex=False).str.strip()
    )
    yield_data[column] = pd.to_numeric(yield_data[column], errors='coerce')
for column in ['District', 'Season', 'CropCategory', 'Crop']:
    yield_data[column] = yield_data[column].astype('string').str.strip()
yield_data = yield_data.dropna(subset=required).copy()
yield_data = yield_data[(yield_data['Extent'] > 0) & (yield_data['Production'] >= 0)].copy()
yield_data['Yield_t_ha'] = yield_data['Production'] / yield_data['Extent']
yield_data = yield_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['Yield_t_ha'])
yield_data = yield_data[yield_data['Yield_t_ha'] <= yield_data['Yield_t_ha'].quantile(0.995)].copy()
yield_data['Year'] = yield_data['Year'].astype(int)
yield_data = yield_data.sort_values(['Crop', 'District', 'Year', 'Season']).reset_index(drop=True)

prepared_file = PREPROCESSED_DIR / 'wijesinghe_yield_preprocessed_15000plus.csv'
yield_data.to_csv(prepared_file, index=False)
print(f'Raw source rows: {len(raw):,}')
print(f'Prepared yield rows: {len(yield_data):,}')
if len(yield_data) < 15000:
    raise ValueError('The prepared dataset is below the required 15,000 genuine records.')
print(f'Saved prepared dataset: {prepared_file.relative_to(REPO_DIR)}')
display(yield_data.head())


Raw source rows: 94,755
Prepared yield rows: 60,529
Saved prepared dataset: Wijesinghe.D.T.D-Crop Yield Preprocessing/wijesinghe_yield_preprocessed_15000plus.csv


,District,Season,CropCategory,Crop,Year,Extent,Production,Yield_t_ha
0,Ampara,Maha,Low Country Vegetable,Ash Plantain,2001,774.0,5809.0,7.505168
1,Ampara,Total,Low Country Vegetable,Ash Plantain,2001,1025.0,8087.0,7.889756
2,Ampara,Yala,Low Country Vegetable,Ash Plantain,2001,251.0,2278.0,9.075697
3,Ampara,Maha,Low Country Vegetable,Ash Plantain,2002,768.0,5780.0,7.526042
4,Ampara,Total,Low Country Vegetable,Ash Plantain,2002,790.0,5976.0,7.564557


In [3]:
features = ['District', 'Season', 'CropCategory', 'Crop', 'Year', 'Extent']
target = 'Yield_t_ha'
train_data, test_data = train_test_split(yield_data, test_size=0.2, random_state=42)
categorical = ['District', 'Season', 'CropCategory', 'Crop']
numeric = ['Year', 'Extent']
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', 'passthrough', numeric)
])
yield_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, max_depth=18, min_samples_leaf=3, random_state=42, n_jobs=-1))
])
yield_model.fit(train_data[features], train_data[target])
predicted = yield_model.predict(test_data[features])
metrics = pd.DataFrame([{
    'Records': len(yield_data),
    'TrainRecords': len(train_data),
    'TestRecords': len(test_data),
    'MAE_t_per_ha': mean_absolute_error(test_data[target], predicted),
    'RMSE_t_per_ha': np.sqrt(mean_squared_error(test_data[target], predicted)),
    'R2': r2_score(test_data[target], predicted)
}])
metrics.to_csv(OUTPUT_DIR / 'wijesinghe_yield_model_metrics.csv', index=False)
joblib.dump(yield_model, OUTPUT_DIR / 'wijesinghe_yield_model.joblib')

example = pd.DataFrame([{
    'District': 'Colombo',
    'Season': 'Maha',
    'CropCategory': 'Low Country Vegetable',
    'Crop': yield_data['Crop'].mode().iloc[0],
    'Year': int(yield_data['Year'].max()) + 1,
    'Extent': float(yield_data['Extent'].median())
}])
example['Predicted_Yield_t_ha'] = yield_model.predict(example[features])
example.to_csv(OUTPUT_DIR / 'wijesinghe_yield_example_prediction.csv', index=False)
display(metrics.round(4))
display(example)


,Records,TrainRecords,TestRecords,MAE_t_per_ha,RMSE_t_per_ha,R2
0,60529,48423,12106,1.9936,3.0962,0.6756


,District,Season,CropCategory,Crop,Year,Extent,Predicted_Yield_t_ha
0,Colombo,Maha,Low Country Vegetable,Chillies (Green),2026,65.0,10.225759


## National historical-yield comparison



In [4]:
from IPython.display import display

NATIONAL_HISTORY_FILE = REPO_DIR / "Reconstruction Outputs" / "Preprocessed Datasets" / "01_production_preprocessed.csv"
if not NATIONAL_HISTORY_FILE.is_file():
    raise FileNotFoundError(
        "Run Notebook 01 and push its output first: "
        "Reconstruction Outputs/Preprocessed Datasets/01_production_preprocessed.csv"
    )

national_history = pd.read_csv(NATIONAL_HISTORY_FILE)
required_history_columns = {
    "District", "Season", "CropCategory", "Crop", "Year", "Extent", "Yield_t_ha"
}
missing_history_columns = sorted(required_history_columns - set(national_history.columns))
if missing_history_columns:
    raise ValueError(f"Notebook 01 output is missing columns: {missing_history_columns}")

for column in ["Year", "Extent", "Yield_t_ha"]:
    national_history[column] = pd.to_numeric(national_history[column], errors="coerce")
national_history = national_history.dropna(
    subset=["District", "Season", "CropCategory", "Crop", "Year", "Extent", "Yield_t_ha"]
).copy()
national_history["Year"] = national_history["Year"].astype(int)
national_history = national_history.loc[
    national_history["District"].astype(str).str.strip().eq("National Total")
    & national_history["Season"].astype(str).isin(["Maha", "Yala"])
].copy()
if national_history.empty:
    raise ValueError("Notebook 01 output contains no National Total Maha/Yala records.")

# Choose the latest crop/category/season combination available in the 699-row analysis.
selected = national_history.sort_values(["Year", "Crop"]).iloc[-1]
CROP = str(selected["Crop"])
CROP_CATEGORY = str(selected["CropCategory"])
SEASON = str(selected["Season"])
FORECAST_YEAR = int(national_history["Year"].max()) + 1

history_match = national_history.loc[
    national_history["Crop"].astype(str).eq(CROP)
    & national_history["CropCategory"].astype(str).eq(CROP_CATEGORY)
    & national_history["Season"].astype(str).eq(SEASON)
    & national_history["Year"].lt(FORECAST_YEAR)
].copy()
if history_match.empty:
    raise ValueError("No earlier national history matches the selected crop and season.")

# Use the existing Notebook 07 model and its feature schema for the model estimate.
model_rows = yield_data.loc[
    yield_data["Crop"].astype(str).eq(CROP)
    & yield_data["CropCategory"].astype(str).eq(CROP_CATEGORY)
    & yield_data["Season"].astype(str).eq(SEASON)
]
if model_rows.empty:
    raise ValueError(
        "The selected national crop/category/season is not available in this notebook's model data."
    )

planned_extent = float(model_rows["Extent"].median())
model_features = ["District", "Season", "CropCategory", "Crop", "Year", "Extent"]
model_input = pd.DataFrame([{
    "District": "National Total",
    "Season": SEASON,
    "CropCategory": CROP_CATEGORY,
    "Crop": CROP,
    "Year": FORECAST_YEAR,
    "Extent": planned_extent,
}])[model_features]

comparison = pd.DataFrame([{
    "Crop": CROP,
    "CropCategory": CROP_CATEGORY,
    "Season": SEASON,
    "District": "National Total",
    "ForecastYear": FORECAST_YEAR,
    "PlannedExtent_ha": planned_extent,
    "ExistingModel_PredictedYield_t_ha": float(yield_model.predict(model_input)[0]),
    "NationalHistoricalAverageYield_t_ha": float(history_match["Yield_t_ha"].mean()),
    "HistoricalRecordsUsed": len(history_match),
    "HistoryFirstYear": int(history_match["Year"].min()),
    "HistoryLastYear": int(history_match["Year"].max()),
    "HistoricalValueIsModelFeature": False,
}])

print(f"Notebook 01 national-analysis rows available: {len(national_history):,}")
print("Model estimate beside the historical reference (t/ha):")
display(comparison.round(3))
print("The historical mean is a comparison reference, not a model input.")


Notebook 01 national-analysis rows available: 699
Model estimate beside the historical reference (t/ha):


,Crop,CropCategory,Season,District,ForecastYear,PlannedExtent_ha,ExistingModel_PredictedYield_t_ha,NationalHistoricalAverageYield_t_ha,HistoricalRecordsUsed,HistoryFirstYear,HistoryLastYear,HistoricalValueIsModelFeature
0,Winged Bean,Low Country Vegetable,Maha,National Total,2026,38.0,10.226,9.009,13,2013,2025,False


The historical mean is a comparison reference, not a model input.


In [5]:
comparison_file = OUTPUT_DIR / "national_historical_yield_reference_comparison.csv"
comparison.to_csv(comparison_file, index=False)
print("Saved comparison:", comparison_file.relative_to(REPO_DIR))


Saved comparison: Wijesinghe.D.T.D-Crop Yield Outputs/national_historical_yield_reference_comparison.csv


In [ ]:
from getpass import getpass
from google.colab import _message
from urllib.request import Request, urlopen
import json
import os
import subprocess
import tempfile

NOTEBOOK_NAME = "07_Wijesinghe_DTD_Crop_Yield_Preprocessing.ipynb"
FILES_TO_ADD = [
    NOTEBOOK_NAME,
    "Wijesinghe.D.T.D-Crop Yield Preprocessing/wijesinghe_yield_preprocessed_15000plus.csv",
    "Wijesinghe.D.T.D-Crop Yield Outputs/wijesinghe_yield_example_prediction.csv",
    "Wijesinghe.D.T.D-Crop Yield Outputs/wijesinghe_yield_model.joblib",
    "Wijesinghe.D.T.D-Crop Yield Outputs/wijesinghe_yield_model_metrics.csv",
    "Wijesinghe.D.T.D-Crop Yield Outputs/national_historical_yield_reference_comparison.csv",
]

# Save the currently open Colab notebook into the cloned repository.
current = _message.blocking_request("get_ipynb", request="", timeout_sec=15)
if not isinstance(current.get("ipynb"), dict):
    raise RuntimeError("Colab could not retrieve this notebook. Save it and retry.")
(REPO_DIR / NOTEBOOK_NAME).write_text(
    json.dumps(current["ipynb"], ensure_ascii=False, indent=1), encoding="utf-8"
)

missing = [name for name in FILES_TO_ADD if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Run earlier cells first. Missing:\n" + "\n".join(missing))
oversized = [
    name for name in FILES_TO_ADD
    if (REPO_DIR / name).stat().st_size >= 100 * 1024 * 1024
]
if oversized:
    raise RuntimeError("Files exceed GitHub's 100 MB limit:\n" + "\n".join(oversized))

github_username = input("GitHub username: ").strip()
commit_name = input("Git commit name: ").strip()
github_email = input("GitHub email: ").strip()
token = getpass("GitHub Personal Access Token (hidden): ").strip()
if not all([github_username, commit_name, github_email, token]):
    raise ValueError("Username, commit name, email, and token are required.")
if input(f"Confirm you are authorized to push as @{github_username} (type YES): ").strip() != "YES":
    raise RuntimeError("Cancelled; no push was attempted.")

request = Request(
    "https://api.github.com/user",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    },
)
with urlopen(request, timeout=20) as response:
    authenticated_user = json.loads(response.read().decode("utf-8")).get("login", "")
if authenticated_user.casefold() != github_username.casefold():
    raise RuntimeError(f"Token belongs to @{authenticated_user}, not @{github_username}.")

subprocess.run(["git", "config", "user.name", commit_name], cwd=REPO_DIR, check=True)
subprocess.run(["git", "config", "user.email", github_email], cwd=REPO_DIR, check=True)
subprocess.run(["git", "add", "--", *FILES_TO_ADD], cwd=REPO_DIR, check=True)
staged = subprocess.run(
    ["git", "diff", "--cached", "--name-only"], cwd=REPO_DIR,
    text=True, capture_output=True, check=True
).stdout.splitlines()
unexpected = sorted(set(staged) - set(FILES_TO_ADD))
if unexpected:
    raise RuntimeError("Unexpected staged files; cancelled:\n" + "\n".join(unexpected))

if staged:
    print("Files to commit:")
    print("\n".join(staged))
    if input("Type PUSH to commit and push to main: ").strip() != "PUSH":
        raise RuntimeError("Cancelled; no push was attempted.")
    subprocess.run(
        ["git", "commit", "-m", "Add national historical-yield comparison to crop-yield workflow"],
        cwd=REPO_DIR, check=True
    )
else:
    print("No new changes; attempting to push any pending local commit.")

fd, askpass_name = tempfile.mkstemp(prefix="git-askpass-")
os.close(fd)
askpass = Path(askpass_name)
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "$GITHUB_USERNAME" ;;\n'
    '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
    'esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
env = os.environ.copy()
env.update({
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "GITHUB_USERNAME": github_username,
    "GITHUB_TOKEN": token,
})
try:
    result = subprocess.run(
        ["git", "push", "origin", "main"], cwd=REPO_DIR, env=env,
        capture_output=True, text=True
    )
    print(result.stdout or "(no standard output)")
    print(result.stderr or "(no Git error text returned)")
    if result.returncode != 0:
        raise RuntimeError(f"Push failed with exit code {result.returncode}.")
    print(f"Push completed as @{authenticated_user}.")
finally:
    token = ""
    env.pop("GITHUB_TOKEN", None)
    askpass.unlink(missing_ok=True)
